In [0]:
from pyspark.sql.functions import col, from_json, unbase64
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

df_raw = spark.readStream \
  .format("cloudFiles") \
  .option("cloudFiles.format", "json") \
  .option("cloudFiles.schemaLocation", "abfss://bronze@stlynxionzwcio.dfs.core.windows.net/schema/bronze_stream") \
  .load("abfss://raw@stlynxionzwcio.dfs.core.windows.net/iot-lynxion-zwcio/")

df_decoded = df_raw.withColumn("json_str", unbase64(col("Body")).cast("string"))

schema = StructType([
    StructField("event_id", StringType()),
    StructField("device_id", StringType()),
    StructField("timestamp", StringType()),
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
    StructField("speed_kmh", DoubleType()),
    StructField("engine_temp_c", DoubleType()),
    StructField("battery_pct", DoubleType()),
    StructField("status", StringType())
])

df_parsed = df_decoded \
  .withColumn("data", from_json(col("json_str"), schema)) \
  .select("data.*", col("EnqueuedTimeUtc").alias("ingest_timestamp"))

df_parsed.writeStream \
  .format("delta") \
  .option("checkpointLocation", "abfss://bronze@stlynxionzwcio.dfs.core.windows.net/checkpoint/bronze_stream") \
  .trigger(availableNow=True) \
  .toTable("bronze_telemetry_stream")